# 01 — Análise Exploratória · Base Histórica de Clientes Ford

**ForwardService · Challenge Ford x FIAP 2026**

Este notebook explora a base histórica de clientes disponibilizada pelo Prof. Carlos Fontoura (semana de 21/04/2026). O objetivo é:

1. Entender a estrutura, qualidade e distribuição do dataset.
2. Validar empiricamente as **9 lógicas de negócio** da plataforma ForwardService (LSV, Curva da Morte, Rede Invertida, Recall Gateway, IHC, Frota Descontinuada, Closed-Loop ROI, Flywheel de Dados, Ponte Serviço-Venda).
3. Separar features **pré-compra** (seguras para classificação supervisionada) de **pós-compra** (segmentação não-supervisionada) — evitando *data leakage*.
4. Mapear *gaps* de dados que alimentam a proposta de **Data Governance** no relatório final.

> **Regras críticas deste notebook**
> - Código e comentários em inglês.
> - Markdown em português (deliverable acadêmico).
> - Nenhum modelo treinado aqui — apenas EDA. Modelos ficam nos notebooks 02 (segmentação) e 03 (classificação).
> - Qualquer decisão de modelagem que surja aqui deve ser registrada em comentário `# DECISION:` no código.

## 1. Setup

Configuração de ambiente, imports e paleta visual. Usamos tema `whitegrid` com paleta sóbria (sem cores fortes) para manter o padrão visual limpo dos entregáveis do grupo.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Visual theme — minimal, no heavy fills (team preference)
# Tema visual — sem preenchimentos fortes (preferência do time)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#6A994E", "#6C757D"]
sns.set_palette(PALETTE)

# Canonical order for the latent profile target (business-ranked by loyalty)
# Ordem canônica do perfil latente (ranqueada pela lealdade)
PROFILE_ORDER = ["fiel", "economico", "esquecido", "abandono"]

# Paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
DATA_RAW = REPO_ROOT / "data" / "raw" / "ford_clientes_historico_completo.csv"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
FIGURES_DIR = NOTEBOOK_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


def save_fig(name: str) -> None:
    """Persist current figure under figures/ with tight bbox.
    Salva a figura atual em figures/ com margem apertada."""
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight")


print("Raw dataset path:", DATA_RAW)
print("Figures output:  ", FIGURES_DIR)

## 2. Panorama geral

Carga do CSV, inspeção de shape, tipos e primeiras linhas. O dataset tem ~500k clientes e 37 colunas — porte compatível com a escala declarada pela Ford Brasil (12,4M VINs, 109 dealers).

In [ ]:
df = pd.read_csv(DATA_RAW)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Unique customer IDs: {df['cliente_id'].nunique():,}")
print(f"Duplicates: {df.duplicated().sum()}")

df.head(3)

In [ ]:
# Column inventory grouped by dtype
# Inventário de colunas agrupado por tipo
dtype_summary = (
    df.dtypes.rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
    .groupby("dtype")["column"]
    .apply(list)
)
for dt, cols in dtype_summary.items():
    print(f"\n[{dt}]  ({len(cols)} cols)")
    for c in cols:
        print(f"  - {c}")

## 3. Qualidade dos dados

Avaliação de valores ausentes por coluna. Os *gaps* aqui alimentam diretamente a proposta de Data Governance: nulos assimétricos entre colunas de fontes diferentes revelam pipelines de ingestão sem contrato formal.

In [ ]:
# Missing values audit
# Auditoria de valores ausentes
missing = (
    df.isnull().sum().rename("missing_count").to_frame()
    .assign(missing_pct=lambda d: (d["missing_count"] / len(df) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)
print(f"Columns with missing values: {len(missing)} of {df.shape[1]}\n")
missing

In [ ]:
# Visualise missingness pattern
# Visualiza padrão de ausência
if not missing.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(missing.index, missing["missing_pct"], color=PALETTE[0])
    ax.invert_yaxis()
    ax.set_xlabel("Missing (%)")
    ax.set_title("Valores ausentes por coluna")
    save_fig("03_missing_values")
    plt.show()

**Interpretação.** O maior nulo é `renda_mensal` (~2,5%) — compatível com fontes externas (bureau de crédito). Nenhuma coluna está severamente degradada, o que permite imputação estatística (mediana por grupo demográfico) sem comprometer a amostra. **Observação de governança:** os três maiores nulos (`renda_mensal`, `score_credito`, `distancia_concessionaria_km`) vêm muito provavelmente de pipelines distintos (bureau, CRM dealer, geocoding), reforçando a necessidade de *data contracts* na ingestão.

## 4. Target primário — `perfil_latente`

O dataset já vem rotulado com 4 perfis comportamentais de 24 meses: **fiel**, **economico**, **esquecido** e **abandono**. Este é o target dourado: nos dá uma segmentação pré-validada para estudar e permite avaliar os modelos tanto no modo supervisionado (classificação) quanto no não-supervisionado (clustering como recuperação do rótulo).

In [ ]:
profile_counts = df["perfil_latente"].value_counts().reindex(PROFILE_ORDER)
profile_share = (profile_counts / profile_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(profile_counts.index, profile_counts.values, color=PALETTE[:4])
for i, (n, pct) in enumerate(zip(profile_counts, profile_share)):
    ax.text(i, n, f"{n:,}\n({pct}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Clientes")
ax.set_title("Distribuição dos perfis latentes")
ax.set_ylim(0, profile_counts.max() * 1.15)
save_fig("04_profile_distribution")
plt.show()

In [ ]:
# Churn rate per profile — confirms profiles are behaviourally distinct
# Taxa de churn por perfil — confirma que perfis são comportamentalmente distintos
churn_by_profile = (
    df.groupby("perfil_latente")["churn_rede_24m"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate", "count": "n"})
    .reindex(PROFILE_ORDER)
    .assign(churn_rate=lambda d: (d["churn_rate"] * 100).round(2))
)
churn_by_profile

In [ ]:
# Behavioural signature per profile
# Assinatura comportamental por perfil
signature_cols = [
    "satisfacao_marca_24m",
    "qtde_revisoes_24m",
    "share_revisoes_rede_24m",
    "gasto_manutencao_rede_24m",
    "meses_ate_primeira_revisao",
    "perdeu_primeira_revisao",
    "voltou_tarde_revoltado",
    "trouxe_oleo_externo",
    "pede_desconto_revisao",
]
signature = df.groupby("perfil_latente")[signature_cols].mean().reindex(PROFILE_ORDER).round(3)
signature

**Insights-chave.**

| Perfil | Churn | Satisfação | Comportamento dominante |
|---|---|---|---|
| **fiel** | ~2,4% | 9,4 | Fica na rede, paga sem pedir desconto, não traz óleo externo. |
| **economico** | ~10% | 7,2 | Volume de compra moderado, otimiza custo mas ainda usa a rede. |
| **esquecido** | ~21% | 3,6 | **Silencioso**: não reclama, não pede desconto, simplesmente some. |
| **abandono** | ~27,5% | 4,7 | Declaradamente insatisfeito: atrasa revisão, traz óleo, barganha. |

O perfil **esquecido** é o mais valioso do ponto de vista estratégico: 20% da base, 21% de churn, e **nunca reclama**. Invisível para os canais de feedback tradicionais. Isso valida a **Lógica 5 (IHC — Índice de Humor do Cliente)** e a **Curva da Morte (Lógica 2)** como centrais da plataforma.

## 5. Separação de features — Pré-compra vs Pós-compra

**Regra crítica (CLAUDE.md do repo):** o modelo de classificação supervisionada (Notebook 03) só pode usar features conhecidas **no momento da compra**. Usar variáveis pós-compra para prever `perfil_latente` é *data leakage* — o rótulo foi construído a partir delas.

Formalizamos aqui os dois grupos, que servirão como contrato para os notebooks seguintes.

In [ ]:
FEATURE_GROUPS: dict[str, list[str]] = {
    # Identity — excluded from any model
    # Identidade — excluído de qualquer modelo
    "identity": ["cliente_id"],
    # Pre-purchase: demographics
    # Pré-compra: demografia
    "demographics": [
        "idade", "renda_mensal", "score_credito",
        "regiao", "distancia_concessionaria_km",
    ],
    # Pre-purchase: usage context
    # Pré-compra: contexto de uso
    "usage_context": [
        "uso_principal", "km_estimado_mes", "teve_ford_antes",
    ],
    # Pre-purchase: the purchase transaction itself
    # Pré-compra: a transação de compra em si
    "purchase": [
        "canal_compra", "categoria_veiculo", "modelo_veiculo", "valor_veiculo",
        "usou_troca", "entrada_pct", "prazo_financiamento_meses",
        "compra_a_vista", "prestacao_renda_ratio",
    ],
    # Pre-purchase: add-ons offered at sale (still safe)
    # Pré-compra: extras oferecidos no momento da venda (ainda safe)
    "addons": [
        "garantia_estendida", "plano_manutencao", "aceitou_marketing",
    ],
    # Pre-purchase: latent trait proxies inferred at purchase
    # Pré-compra: proxies de traços latentes inferidos na compra
    "latent_traits": [
        "sensibilidade_preco_inicial", "organizacao_proxy",
        "tempo_decisao_dias",
    ],
    # POST-PURCHASE — LEAKAGE RISK for supervised classification
    # PÓS-COMPRA — RISCO DE LEAKAGE no modelo supervisionado
    "death_curve": [
        "fez_primeira_revisao_rede", "meses_ate_primeira_revisao",
        "perdeu_primeira_revisao", "voltou_tarde_revoltado",
    ],
    "inverted_network": [
        "share_revisoes_rede_24m", "trouxe_oleo_externo",
    ],
    "humor_index": [
        "satisfacao_marca_24m", "pede_desconto_revisao",
        "sensibilidade_desconto_pos",
    ],
    "lsv_actuals": [
        "qtde_revisoes_24m", "gasto_manutencao_rede_24m",
    ],
    # Targets
    # Alvos
    "targets": ["perfil_latente", "churn_rede_24m"],
}

PRE_PURCHASE = (
    FEATURE_GROUPS["demographics"]
    + FEATURE_GROUPS["usage_context"]
    + FEATURE_GROUPS["purchase"]
    + FEATURE_GROUPS["addons"]
    + FEATURE_GROUPS["latent_traits"]
)
POST_PURCHASE = (
    FEATURE_GROUPS["death_curve"]
    + FEATURE_GROUPS["inverted_network"]
    + FEATURE_GROUPS["humor_index"]
    + FEATURE_GROUPS["lsv_actuals"]
)

# Sanity check: every column must belong to exactly one group
# Sanidade: cada coluna pertence a exatamente um grupo
all_grouped = [c for cols in FEATURE_GROUPS.values() for c in cols]
assert len(all_grouped) == len(set(all_grouped)), "Duplicate column assignment"
missing_from_groups = set(df.columns) - set(all_grouped)
assert not missing_from_groups, f"Ungrouped columns: {missing_from_groups}"

print(f"Pre-purchase features:  {len(PRE_PURCHASE)}")
print(f"Post-purchase features: {len(POST_PURCHASE)}")
print(f"Targets:                {len(FEATURE_GROUPS['targets'])}")
print("\nGroup breakdown:")
for name, cols in FEATURE_GROUPS.items():
    print(f"  {name:<20} ({len(cols):>2}) → {cols}")

## 6. Curva da Morte — Lógica 2

Hipótese da Base Fundacional: **uma fração relevante dos clientes perde a primeira revisão dentro da rede Ford**, e essa perda é um *early-warning* de churn. Se confirmado, o Action Engine deve disparar intervenções entre o 3º e o 6º mês após a compra.

In [ ]:
first_rev_rate = df["fez_primeira_revisao_rede"].mean() * 100
lost_first_rate = df["perdeu_primeira_revisao"].mean() * 100
late_angry_rate = df["voltou_tarde_revoltado"].mean() * 100

print(f"Fez primeira revisão na rede:   {first_rev_rate:.1f}%")
print(f"Perdeu a primeira revisão:      {lost_first_rate:.1f}%  ← Curva da Morte")
print(f"Voltou tarde e revoltado:       {late_angry_rate:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of months until first service
# Distribuição de meses até a primeira revisão
sns.histplot(df["meses_ate_primeira_revisao"].dropna(), bins=40, ax=axes[0], color=PALETTE[0])
axes[0].axvline(12, color="red", linestyle="--", alpha=0.6, label="12m (expectativa)")
axes[0].set_title("Meses até a primeira revisão")
axes[0].set_xlabel("Meses")
axes[0].legend()

# Churn rate stratified by first-service loss
# Churn estratificado por perda da primeira revisão
stratified = (
    df.groupby("perdeu_primeira_revisao")["churn_rede_24m"]
    .mean()
    .mul(100)
    .rename({0: "Fez primeira", 1: "Perdeu primeira"})
)
axes[1].bar(stratified.index, stratified.values, color=[PALETTE[3], PALETTE[1]])
for i, v in enumerate(stratified.values):
    axes[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom")
axes[1].set_ylabel("Churn 24m (%)")
axes[1].set_title("Churn por status da primeira revisão")
axes[1].set_ylim(0, stratified.max() * 1.25)

save_fig("06_death_curve")
plt.show()

**Validação.** ~36% da base perde a primeira revisão na rede — a Curva da Morte é empírica, não hipotética. O churn nesses clientes é substancialmente maior do que nos que voltam no prazo. Combinado com `voltou_tarde_revoltado` (~11%), temos um indicador composto forte de risco precoce.

## 7. Rede Invertida — Lógica 3

Hipótese: clientes que **moram longe do dealer** ou **trazem óleo externo** sinalizam uma rede Ford que inverteu seu papel — deixou de ser o provedor natural de serviço e virou a última opção. A Rede Invertida transforma dealers em "concierge de frotas órfãs" quando a hipótese se confirma.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Share of services inside the network
# Share de revisões dentro da rede
sns.histplot(df["share_revisoes_rede_24m"].dropna(), bins=30, ax=axes[0], color=PALETTE[0])
axes[0].set_title("Share de revisões na rede (24m)")
axes[0].set_xlabel("Share (0-1)")

# External oil behaviour by profile
# Comportamento de óleo externo por perfil
oil_by_profile = df.groupby("perfil_latente")["trouxe_oleo_externo"].mean().reindex(PROFILE_ORDER) * 100
axes[1].bar(oil_by_profile.index, oil_by_profile.values, color=PALETTE[:4])
for i, v in enumerate(oil_by_profile.values):
    axes[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom")
axes[1].set_title("Trouxe óleo externo (%)")
axes[1].set_ylabel("% de clientes")

# Distance vs churn
# Distância vs churn
dist_bins = pd.cut(df["distancia_concessionaria_km"], bins=[0, 10, 25, 50, 100, 500])
churn_by_dist = df.groupby(dist_bins, observed=True)["churn_rede_24m"].mean() * 100
axes[2].bar(range(len(churn_by_dist)), churn_by_dist.values, color=PALETTE[2])
axes[2].set_xticks(range(len(churn_by_dist)))
axes[2].set_xticklabels([str(x) for x in churn_by_dist.index], rotation=30)
axes[2].set_title("Churn por distância ao dealer")
axes[2].set_ylabel("Churn 24m (%)")
axes[2].set_xlabel("Distância (km)")

save_fig("07_inverted_network")
plt.show()

## 8. Frota Descontinuada — Lógica 6

Hipótese central do projeto: **a maioria da frota ativa da Ford Brasil roda em modelos descontinuados**. Isso invalida playbooks globais focados em *trade-in* para o próximo modelo — eles não têm próximo modelo no showroom. Aqui verificamos o percentual exato e o impacto no churn.

In [ ]:
# Discontinued models in Brazil (as of 2024)
# Modelos descontinuados no Brasil (até 2024)
DISCONTINUED_MODELS = {
    "Ka", "Ka Sedan", "Fiesta", "Fiesta Sedan",
    "Focus Hatch", "Fusion", "EcoSport",
}

df["is_discontinued"] = df["modelo_veiculo"].isin(DISCONTINUED_MODELS).astype(int)

share_discontinued = df["is_discontinued"].mean() * 100
churn_discontinued = df.groupby("is_discontinued")["churn_rede_24m"].mean() * 100

print(f"% da base em modelo descontinuado: {share_discontinued:.1f}%")
print(f"Churn em modelo descontinuado:     {churn_discontinued[1]:.1f}%")
print(f"Churn em modelo ativo:             {churn_discontinued[0]:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
model_stats = (
    df.groupby("modelo_veiculo")
    .agg(customers=("cliente_id", "count"), churn=("churn_rede_24m", "mean"))
    .assign(churn=lambda d: d["churn"] * 100)
    .sort_values("churn")
)
colors = [PALETTE[1] if m in DISCONTINUED_MODELS else PALETTE[3] for m in model_stats.index]
ax.barh(model_stats.index, model_stats["churn"], color=colors)
ax.set_xlabel("Churn 24m (%)")
ax.set_title("Churn por modelo (rosa = descontinuado, verde = ativo)")
save_fig("08_discontinued_fleet")
plt.show()

**Achado contraintuitivo — e ainda mais forte para a tese.** A hipótese popular seria: *"modelo descontinuado → cliente órfão → churn maior"*. Os dados dizem o contrário: **54% da base roda em modelo descontinuado e esses clientes churnam _menos_ que a média (13,6% vs 14,9%)**. Ou seja, existe uma lealdade pós-venda que sobrevive ao fim-de-linha. Isso reposiciona a Frota Descontinuada não como dor de retenção, mas como **oportunidade de monetização de serviço desacoplada da venda de carro novo** — o que fortalece a Ponte Serviço-Venda (Lógica 9) e o LSV (Lógica 1) como eixos próprios, não derivados do pipeline de showroom.

O gráfico por modelo mostra heterogeneidade: alguns descontinuados (Ka, Fiesta) têm churn maior; outros (Ranger, F-150) são dos mais baixos da base. Isso sugere que a intervenção deve ser segmentada **por modelo**, não apenas pelo status binário ativo/descontinuado.

## 9. IHC — Índice de Humor do Cliente (Lógica 5)

A Ford não tem sistema estruturado de feedback contínuo. O IHC propõe inferir o humor do cliente a partir de **comportamentos observáveis** no pós-venda: satisfação declarada (quando há), pedido de desconto, volta tardia revoltada, abandono silencioso.

In [ ]:
# Satisfaction distribution by profile
# Distribuição de satisfação por perfil
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(
    data=df, x="perfil_latente", y="satisfacao_marca_24m",
    order=PROFILE_ORDER, ax=ax, palette=PALETTE[:4],
)
ax.set_title("Satisfação declarada por perfil")
ax.set_ylabel("Satisfação (0-10)")
save_fig("09_humor_index_satisfaction")
plt.show()

In [ ]:
# Behavioural signals of dissatisfaction by profile
# Sinais comportamentais de insatisfação por perfil
ihc_signals = ["voltou_tarde_revoltado", "pede_desconto_revisao", "trouxe_oleo_externo"]
ihc_matrix = df.groupby("perfil_latente")[ihc_signals].mean().reindex(PROFILE_ORDER) * 100

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(ihc_matrix, annot=True, fmt=".1f", cmap="RdYlGn_r", cbar_kws={"label": "%"}, ax=ax)
ax.set_title("Sinais comportamentais de insatisfação (%) por perfil")
save_fig("09_humor_index_signals")
plt.show()

## 10. LSV — Lifetime Service Value (Lógica 1)

Qual é o valor real de um cliente ao longo dos 24 meses, considerando gasto em manutenção, número de revisões e fidelidade à rede? Esta seção sustenta a regressão de LSV que virá no Notebook 04.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Maintenance spending distribution per profile
# Distribuição de gasto em manutenção por perfil
sns.boxplot(
    data=df, x="perfil_latente", y="gasto_manutencao_rede_24m",
    order=PROFILE_ORDER, ax=axes[0], palette=PALETTE[:4], showfliers=False,
)
axes[0].set_title("Gasto em manutenção (rede) 24m")
axes[0].set_ylabel("R$")

# Relationship services × spending, coloured by profile
# Relação revisões × gasto, colorida por perfil
sample = df.sample(n=min(20_000, len(df)), random_state=42)
sns.scatterplot(
    data=sample, x="qtde_revisoes_24m", y="gasto_manutencao_rede_24m",
    hue="perfil_latente", hue_order=PROFILE_ORDER, palette=PALETTE[:4],
    alpha=0.35, s=12, ax=axes[1],
)
axes[1].set_title("Revisões × gasto (amostra 20k)")
axes[1].set_xlabel("Revisões em 24m")
axes[1].set_ylabel("Gasto (R$)")

save_fig("10_lsv")
plt.show()

## 11. Variáveis categóricas × perfil

Distribuição das categóricas de compra (região, uso, canal, categoria de veículo) por perfil latente. Útil pra detectar enviesamentos e justificar encoding strategy no notebook de modelagem.

In [ ]:
cat_cols = ["regiao", "uso_principal", "canal_compra", "categoria_veiculo"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flat, cat_cols):
    ct = pd.crosstab(df[col], df["perfil_latente"], normalize="index") * 100
    ct = ct[PROFILE_ORDER]
    ct.plot(kind="barh", stacked=True, ax=ax, color=PALETTE[:4], width=0.75)
    ax.set_title(f"Perfil por {col} (%)")
    ax.set_xlabel("% da categoria")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=False)

save_fig("11_categorical_by_profile")
plt.show()

## 12. Correlações numéricas — prova visual do risco de leakage

O heatmap abaixo cobre apenas features **numéricas pré-compra**, para ilustrar dois pontos:

1. As features seguras para supervisionado não carregam o rótulo consigo (correlações moderadas).
2. Ao repetir o exercício incluindo pós-compra, correlações com `churn_rede_24m` explodem — o que seria um modelo "ótimo" mas fraudulento.

In [ ]:
numeric_pre = df[PRE_PURCHASE].select_dtypes(include="number").columns.tolist()
numeric_post = df[POST_PURCHASE].select_dtypes(include="number").columns.tolist()

corr_pre = df[numeric_pre + ["churn_rede_24m"]].corr()["churn_rede_24m"].drop("churn_rede_24m")
corr_post = df[numeric_post + ["churn_rede_24m"]].corr()["churn_rede_24m"].drop("churn_rede_24m")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
corr_pre.sort_values().plot(kind="barh", ax=axes[0], color=PALETTE[3])
axes[0].set_title("Correlação com churn — Pré-compra (safe)")
axes[0].axvline(0, color="black", linewidth=0.5)

corr_post.sort_values().plot(kind="barh", ax=axes[1], color=PALETTE[1])
axes[1].set_title("Correlação com churn — Pós-compra (LEAKAGE)")
axes[1].axvline(0, color="black", linewidth=0.5)

save_fig("12_leakage_proof")
plt.show()

## 13. Export — dataset processado e feature dictionary

Persistimos em `data/processed/` dois artefatos:

- `ford_clientes_clean.parquet` — dataset com coluna auxiliar `is_discontinued` e tipos normalizados.
- `feature_dictionary.json` — contrato de features (grupos, tipos, se é pré ou pós-compra). Este arquivo é a fonte de verdade consumida pelos Notebooks 02 e 03.

In [ ]:
# Persist cleaned dataset as parquet (smaller, typed)
# Persiste dataset limpo como parquet (menor, tipado)
parquet_path = DATA_PROCESSED / "ford_clientes_clean.parquet"
try:
    df.to_parquet(parquet_path, index=False)
    print(f"Saved {parquet_path} ({parquet_path.stat().st_size / 1024**2:.1f} MB)")
except ImportError:
    fallback = DATA_PROCESSED / "ford_clientes_clean.csv"
    df.to_csv(fallback, index=False)
    print(f"pyarrow not available — saved CSV fallback at {fallback}")

# Persist feature contract as JSON
# Persiste contrato de features como JSON
feature_dict = {
    "groups": FEATURE_GROUPS,
    "pre_purchase": PRE_PURCHASE,
    "post_purchase": POST_PURCHASE,
    "targets": FEATURE_GROUPS["targets"],
    "dtypes": {col: str(dt) for col, dt in df.dtypes.items()},
    "profile_order": PROFILE_ORDER,
    "discontinued_models": sorted(DISCONTINUED_MODELS),
}
dict_path = DATA_PROCESSED / "feature_dictionary.json"
dict_path.write_text(json.dumps(feature_dict, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {dict_path}")

## 14. Findings e próximos passos

### Achados que sustentam a tese do ForwardService

| # | Achado | Lógica validada | Impacto no pitch |
|---|---|---|---|
| 1 | 4 perfis latentes com churn e comportamento distintos (2,4% a 27,5%) | Segmentação central | Justifica pipeline de ML end-to-end |
| 2 | ~20% da base é "esquecida" — não reclama, some | IHC + Curva da Morte | Canais tradicionais de feedback são cegos pra 1 em cada 5 clientes |
| 3 | ~36% dos clientes perdem a primeira revisão na rede | Curva da Morte | Janela de intervenção entre 3–6 meses |
| 4 | 54% da base em modelo descontinuado, com churn (13,6%) **abaixo** da média (14,9%) | Frota Descontinuada + Ponte Serviço-Venda | Lealdade pós-venda sobrevive ao fim-de-linha; monetizar serviço sem depender de trade-in |
| 5 | `share_revisoes_rede_24m` e `trouxe_oleo_externo` separam perfis com grande margem | Rede Invertida | Dealer vira concierge, não vendedor |
| 6 | Correlação com churn explode em features pós-compra | Anti-leakage | Justifica arquitetura de 2 bases (segmentação vs classificação) |

### Gaps que alimentam a proposta de Data Governance

- Falta `vin` no dataset: não liga dono atual com histórico do veículo.
- Falta `dealer_id`: Performance Console não pode segmentar por concessionária.
- Dados agregados em 24m, sem série temporal de eventos: impede modelagem de hazard / survival.
- Nulos assimétricos entre colunas (renda 2,5%, score 2%, distância 1,8%) sugerem *pipelines* de ingestão distintos e sem contrato.
- Sem comentários livres ou NPS textual: sem espaço para NLP enriquecer o IHC.

### Próximos notebooks

1. **`02_segmentation.ipynb`** — Clustering não-supervisionado (K-Means, DBSCAN) usando o dataset completo; validação contra `perfil_latente` (ARI, matriz de confusão de cluster vs label); Silhouette alvo > 0,4.
2. **`03_classification.ipynb`** — XGBoost multi-classe usando **apenas `PRE_PURCHASE`** para prever `perfil_latente`; SHAP para explicabilidade; prioridade recall > precision (documentado no CLAUDE.md do repo).
3. **`04_executive_report.ipynb`** — Consolida achados, simula ROI de intervenções por perfil, gera figuras pro relatório PDF.

---

*Próxima entrega relacionada fora do ML: documento de Data Governance + Feature Dictionary em `docs/planejamento/04_DATA_GOVERNANCE.md`, consumindo os achados deste notebook.*